# Deep Past - Pretrained Seq2Seq Baseline (Modular)

This notebook is a **complete baseline** for immediate Kaggle submission using a pretrained encoder-decoder model.

## What this notebook does
- Loads `train.csv` and `test.csv`
- Applies deterministic transliteration/translation normalization
- Builds train/valid split (reproducible)
- Fine-tunes a pretrained seq2seq model (`google/mt5-small` by default)
- Computes competition-aligned dev metric: $\sqrt{BLEU \cdot chrF++}$
- Generates `submission.csv` for Kaggle

> Designed to be modular: each stage can be swapped/improved independently.

## Pipeline Overview
1. **Environment & imports**
2. **Config** (paths, model, hyperparameters)
3. **Data loading utilities**
4. **Normalization utilities**
5. **Split & preprocessing**
6. **Metric function**
7. **Model/tokenizer setup**
8. **Train + evaluate**
9. **Inference + submission file**

In [ ]:
# Optional: uncomment if running in a fresh environment
# %pip install -q transformers datasets sacrebleu sentencepiece accelerate

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

import sacrebleu
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [ ]:
@dataclass
class CFG:
    model_name: str = "google/mt5-small"
    max_source_length: int = 256
    max_target_length: int = 128
    train_batch_size: int = 8
    eval_batch_size: int = 8
    learning_rate: float = 3e-5
    num_train_epochs: int = 2
    weight_decay: float = 0.01
    warmup_steps: int = 100
    gradient_accumulation_steps: int = 2
    valid_ratio: float = 0.1
    split_salt: str = "deep_past_seq2seq_v1"
    seed: int = 42
    output_dir: str = "./outputs_mt5_small"
    predict_with_generate: bool = True
    force_cpu: bool = False

cfg = CFG()

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(cfg.seed)
print(cfg)

In [ ]:
# ---------- Data Loading Utilities ----------
def find_file(filename: str) -> Optional[Path]:
    roots = [Path('/kaggle/input'), Path('../input'), Path('data/raw'), Path('.')]
    matches: List[Path] = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        return None
    matches = sorted(matches, key=lambda p: (len(str(p)), str(p)))
    return matches[0]

def load_core_data() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_path = find_file('train.csv')
    test_path = find_file('test.csv')

    print('train.csv ->', train_path)
    print('test.csv  ->', test_path)

    if train_path is None or test_path is None:
        raise FileNotFoundError('Missing train.csv or test.csv in Kaggle input / local folders.')

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    return train_df, test_df

train_df, test_df = load_core_data()
print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
display(train_df.head(2))
display(test_df.head(2))

In [ ]:
# ---------- Text Normalization Utilities ----------
SUBSCRIPT_MAP = str.maketrans({
    '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
    '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
    'ₓ': 'x',
})

def normalize_transliteration(text: str) -> str:
    if pd.isna(text):
        return ''

    value = str(text)
    value = value.replace('Ḫ', 'H').replace('ḫ', 'h')

    value = re.sub(r'\[x\]', ' __GAP__ ', value)
    value = re.sub(r'\[\s*\.\.\.\s*\]', ' __BIG_GAP__ ', value)
    value = re.sub(r'\.\.\.', ' __BIG_GAP__ ', value)

    value = value.translate(SUBSCRIPT_MAP)

    value = re.sub(r'[!?/]', ' ', value)
    value = value.replace(':', ' ').replace('.', ' ')
    value = value.replace('[', '').replace(']', '')
    value = value.replace('<', '').replace('>', '')

    value = value.replace('__GAP__', '<gap>')
    value = value.replace('__BIG_GAP__', '<big_gap>')

    return re.sub(r'\s+', ' ', value).strip()

def normalize_translation(text: str) -> str:
    if pd.isna(text):
        return ''
    value = str(text)
    value = value.replace('<', '').replace('>', '')
    return re.sub(r'\s+', ' ', value).strip()

In [ ]:
# ---------- Split & Preprocess Utilities ----------
def deterministic_split_by_id(df: pd.DataFrame, id_col: str, valid_ratio: float, salt: str) -> pd.DataFrame:
    cutoff = int(valid_ratio * 10_000)

    def is_valid(identifier: str) -> int:
        payload = f'{salt}:{identifier}'.encode('utf-8')
        bucket = int(hashlib.md5(payload).hexdigest(), 16) % 10_000
        return int(bucket < cutoff)

    out = df.copy()
    out['is_valid'] = out[id_col].astype(str).map(is_valid)
    out['split'] = out['is_valid'].map({0: 'train', 1: 'valid'})
    return out

def build_training_frames(train_raw: pd.DataFrame, test_raw: pd.DataFrame, cfg: CFG):
    train = train_raw.copy()
    test = test_raw.copy()

    train['source_text'] = train['transliteration'].map(normalize_transliteration)
    train['target_text'] = train['translation'].map(normalize_translation)

    test['source_text'] = test['transliteration'].map(normalize_transliteration)

    id_col = 'oare_id' if 'oare_id' in train.columns else train.columns[0]
    train = deterministic_split_by_id(train, id_col=id_col, valid_ratio=cfg.valid_ratio, salt=cfg.split_salt)

    train_part = train[train['split'] == 'train'].reset_index(drop=True)
    valid_part = train[train['split'] == 'valid'].reset_index(drop=True)

    return train, train_part, valid_part, test

train_all_df, train_part_df, valid_part_df, test_ready_df = build_training_frames(train_df, test_df, cfg)

print('train part:', train_part_df.shape)
print('valid part:', valid_part_df.shape)
print('test part :', test_ready_df.shape)
display(train_part_df[['source_text', 'target_text']].head(2))

In [ ]:
# ---------- Competition-Aligned Metric ----------
def geometric_mean_bleu_chrf(references: Iterable[str], hypotheses: Iterable[str]) -> Dict[str, float]:
    refs = list(references)
    hyps = list(hypotheses)

    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
    gmean = math.sqrt(max(bleu, 0.0) * max(chrf, 0.0))

    return {
        'bleu': float(bleu),
        'chrfpp': float(chrf),
        'geometric_mean': float(gmean),
    }

In [ ]:
# ---------- Tokenizer & Model ----------
def detect_device(force_cpu: bool = False) -> str:
    if force_cpu:
        return 'cpu'
    if not torch.cuda.is_available():
        return 'cpu'

    try:
        _ = torch.tensor([1.0], device='cuda') * 2.0
        torch.cuda.synchronize()
        return 'cuda'
    except Exception as exc:
        print('CUDA preflight failed, fallback to CPU:', repr(exc))
        return 'cpu'

device = detect_device(force_cpu=cfg.force_cpu)
print('Using device:', device)

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name)
model.to(device)

print('Loaded model:', cfg.model_name)

In [ ]:
# ---------- Dataset Building ----------
def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[['source_text', 'target_text']], preserve_index=False)

train_ds = to_hf_dataset(train_part_df)
valid_ds = to_hf_dataset(valid_part_df)

def preprocess_batch(examples: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    model_inputs = tokenizer(
        examples['source_text'],
        max_length=cfg.max_source_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples['target_text'],
        max_length=cfg.max_target_length,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_ds.map(preprocess_batch, batched=True, remove_columns=train_ds.column_names)
tokenized_valid = valid_ds.map(preprocess_batch, batched=True, remove_columns=valid_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

print(tokenized_train)
print(tokenized_valid)

In [ ]:
# ---------- Training ----------
use_fp16 = device == 'cuda'

base_kwargs = dict(
    output_dir=cfg.output_dir,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    weight_decay=cfg.weight_decay,
    num_train_epochs=cfg.num_train_epochs,
    warmup_steps=cfg.warmup_steps,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    save_strategy='epoch',
    logging_steps=50,
    predict_with_generate=cfg.predict_with_generate,
    fp16=use_fp16,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    no_cuda=(device == 'cpu'),
)

try:
    training_args = Seq2SeqTrainingArguments(
        eval_strategy='epoch',
        **base_kwargs,
    )
except TypeError:
    training_args = Seq2SeqTrainingArguments(
        evaluation_strategy='epoch',
        **base_kwargs,
    )

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
# ---------- Dev Evaluation ----------
def generate_texts(texts: List[str], batch_size: int = 16, num_beams: int = 4) -> List[str]:
    outputs: List[str] = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=cfg.max_source_length,
        )
        enc = {k: v.to(model.device) for k, v in enc.items()}

        gen_ids = model.generate(
            **enc,
            max_new_tokens=cfg.max_target_length,
            num_beams=num_beams,
            early_stopping=True,
        )
        preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        outputs.extend([p.strip() for p in preds])

    return outputs

valid_preds = generate_texts(valid_part_df['source_text'].tolist(), batch_size=cfg.eval_batch_size)
valid_refs = valid_part_df['target_text'].fillna('').tolist()
scores = geometric_mean_bleu_chrf(valid_refs, valid_preds)
print(json.dumps(scores, indent=2))

In [ ]:
# ---------- Test Inference ----------
test_preds = generate_texts(test_ready_df['source_text'].tolist(), batch_size=cfg.eval_batch_size)
print('Predictions generated:', len(test_preds))
print('Example:', test_preds[0] if len(test_preds) else 'N/A')

In [ ]:
# ---------- Submission ----------
submission = pd.DataFrame({
    'id': test_ready_df['id'],
    'translation': [p if p else 'unknown' for p in test_preds],
})

if submission['id'].duplicated().any():
    raise ValueError('Duplicate ids in submission.')
if submission['translation'].isna().any():
    raise ValueError('Null translations in submission.')

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
display(submission.head())

## Notes for Improvement
- Try `num_train_epochs=3-5` on Kaggle GPU
- Try stronger model (`google/mt5-base`) if GPU memory allows
- Keep preprocessing identical between training and inference
- Add pseudo-parallel data only after this baseline is stable